In [1]:
import os
import sys
import plotly.express as px
import torch as t
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import numpy as np
import einops
from jaxtyping import Int, Float
from typing import List, Optional, Tuple
import functools
from tqdm import tqdm
from IPython.display import display
import webbrowser
import gdown
from transformer_lens.hook_points import HookPoint
from transformer_lens import utils, HookedTransformer, HookedTransformerConfig, FactoredMatrix, ActivationCache
import circuitsvis as cv

# Make sure exercises are in the path
chapter = r"chapter1_transformer_interp"
exercises_dir = Path(f"{os.getcwd().split(chapter)[0]}/{chapter}/exercises").resolve()
section_dir = exercises_dir / "part2_intro_to_mech_interp"
if str(exercises_dir) not in sys.path: sys.path.append(str(exercises_dir))

from plotly_utils import imshow, hist, plot_comp_scores, plot_logit_attribution, plot_loss_difference
from part1_transformer_from_scratch.solutions import get_log_probs
import part2_intro_to_mech_interp.tests as tests

# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)

device = t.device("cuda" if t.cuda.is_available() else "cpu")

MAIN = __name__ == "__main__"

# TransformerLens: Introduction

In [2]:
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

/mnt/share/code/collijk/miniconda/envs/mi/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
# Number of layers = 12
# Heads per layer = 12
# Context window = 1024

gpt2_small.cfg

HookedTransformerConfig:
{'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': 8.0,
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': device(type='cuda'),
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': 0.02886751345948129,
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_value_heads': None,
 'n_layers': 12,
 'n_params': 84934656,
 'normalization_type': 'LNPre',
 'num_experts': None,
 'original_architecture': 'GPT2LMHeadModel',
 'output_logits_soft_cap': -1.0,
 'parallel_attn_mlp': False,
 'positio

In [4]:
model_description_text = '''## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model the model out, let's find the loss on this paragraph!'''

loss = gpt2_small(model_description_text, return_type="loss")
print("Model loss:", loss)

Model loss: tensor(4.3443, device='cuda:0')


In [5]:
gpt2_small.blocks[0].attn.W_V.shape

torch.Size([12, 768, 64])

In [6]:
print(gpt2_small.to_str_tokens("gpt2"))
print(gpt2_small.to_str_tokens(["gpt2", "gpt2"]))
print(gpt2_small.to_tokens("gpt2"))
print(gpt2_small.to_string([50256, 70, 457, 17]))

['<|endoftext|>', 'g', 'pt', '2']
[['<|endoftext|>', 'g', 'pt', '2'], ['<|endoftext|>', 'g', 'pt', '2']]
tensor([[50256,    70,   457,    17]], device='cuda:0')
<|endoftext|>gpt2


In [7]:
logits = gpt2_small(model_description_text, return_type="logits")
prediction = logits.argmax(dim=-1).squeeze()[:-1]
tokens = gpt2_small.to_tokens(model_description_text).squeeze()[1:]
correct = (prediction == tokens).sum()
print(correct / tokens.numel())

tensor(0.2973, device='cuda:0')


In [8]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on taskspecific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_tokens, remove_batch_dim=True)

In [9]:
gpt2_logits.shape

torch.Size([1, 33, 50257])

In [10]:
layer0_pattern_from_cache = gpt2_cache["pattern", 0]

# YOUR CODE HERE - define `layer0_pattern_from_q_and_k` manually, by manually performing the steps of the attention calculation (dot product, masking, scaling, softmax)
q = gpt2_cache["q", 0]
k = gpt2_cache["k", 0]
qk = einops.einsum(q, k, "sq h d, sk h d -> h sq sk") / gpt2_small.cfg.d_head**0.5
mask = t.triu(
    t.ones_like(qk[0]),
    diagonal=1,
).bool()
qk.masked_fill_(mask, -t.inf)
layer0_pattern_from_q_and_k = qk.softmax(-1)


t.testing.assert_close(layer0_pattern_from_cache, layer0_pattern_from_q_and_k)
print("Tests passed!")

Tests passed!


In [12]:
print(type(gpt2_cache))
attention_pattern = gpt2_cache["pattern", 0]
print(attention_pattern.shape)
gpt2_str_tokens = gpt2_small.to_str_tokens(gpt2_text)

# print("Layer 0 Head Attention Patterns:")
# display(cv.attention.attention_heads(
#     tokens=gpt2_str_tokens, 
#     attention=attention_pattern,
#     attention_head_names=[f"L0H{i}" for i in range(12)],
# ))

<class 'transformer_lens.ActivationCache.ActivationCache'>
torch.Size([12, 33, 33])


In [15]:
# neuron_activations_for_all_layers = t.stack([
#     gpt2_cache["post", layer] for layer in range(gpt2_small.cfg.n_layers)
# ], dim=1)

# cv.activations.text_neuron_activations(
#     tokens=gpt2_str_tokens,
#     activations=neuron_activations_for_all_layers
# )

In [17]:
# neuron_activations_for_all_layers_rearranged = utils.to_numpy(einops.rearrange(neuron_activations_for_all_layers, "seq layers neurons -> 1 layers seq neurons"))

# cv.topk_tokens.topk_tokens(
#     # Some weird indexing required here ¯\_(ツ)_/¯
#     tokens=[gpt2_str_tokens], 
#     activations=neuron_activations_for_all_layers_rearranged,
#     max_k=7, 
#     first_dimension_name="Layer", 
#     third_dimension_name="Neuron",
#     first_dimension_labels=list(range(12))
# )

# Finding induction heads

In [ ]:
cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True, # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b", 
    seed=398,
    use_attn_result=True,
    normalization_type=None, # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer"
)

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [ ]:
model = HookedTransformer(cfg)
pretrained_weights = t.load(weights_path, map_location=device)
model.load_state_dict(pretrained_weights)

In [ ]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

In [ ]:
# print(type(cache))
# attention_pattern = cache["pattern", 0]
# print(attention_pattern.shape)
# tokens = model.to_str_tokens(text)

# print("Layer 0 Head Attention Patterns:")
# display(cv.attention.attention_patterns(
#     tokens=tokens, 
#     attention=attention_pattern,
# ))

In [ ]:
# print(type(cache))
# attention_pattern = cache["pattern", 1]
# print(attention_pattern.shape)
# tokens = model.to_str_tokens(text)

# print("Layer 0 Head Attention Patterns:")
# display(cv.attention.attention_heads(
#     tokens=tokens, 
#     attention=attention_pattern,
#     attention_head_names=[f"L1H{i}" for i in range(12)],
# ))

In [ ]:
import matplotlib.pyplot as plt
attn = cache['pattern', 0][0].cpu()
t.diag(attn)

In [ ]:
def attn_detector(cache, criteria):
    detected_heads = []
    nlayers = cache.model.cfg.n_layers
    for layer in range(nlayers):
        attn = cache['pattern', layer]
        heads, *_ = attn.shape
        for head in range(heads):
            head_attn = attn[head]
            if criteria(head_attn):
                detected_heads.append(f"{layer}.{head}")
    return detected_heads
        
def current_attn_detector(cache: ActivationCache) -> List[str]:
    '''
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
    '''
    def _current_attn(attn):
        val = t.diag(attn).mean().item()        
        return val > 0.25
    return attn_detector(cache, _current_attn)

def prev_attn_detector(cache: ActivationCache) -> List[str]:
    '''
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
    '''
    def _prev_attn(attn):
        val = t.diag(attn, diagonal=-1).mean().item()        
        return val > 0.25
    return attn_detector(cache, _prev_attn)

def first_attn_detector(cache: ActivationCache) -> List[str]:
    '''
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    '''
    def _first_attn(attn):
        val = attn[:, 0].mean().item()        
        return val > 0.25
    return attn_detector(cache, _first_attn)


print("Heads attending to current token  = ", ", ".join(current_attn_detector(cache)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(cache)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(cache)))

In [18]:
510100000 * 1000**2 / 40**2 * 4 / 1024**3

318812500000.0